# 🔗 Plantilla universal de cruce y deduplicación · `rues-linker` 0.22.4

Un solo notebook para **dos trabajos**: encontrar la misma empresa en **varias
bases** (record linkage) o encontrar los **duplicados dentro de una** sola
(deduplicación). Viene preconfigurado con el caso **RUES × Exportaciones DANE**,
pero sirve para cualquier par de archivos: los nombres de columna, separadores,
codificaciones y tipos se declaran, no se suponen.

## 🗺️ Mapa del notebook

| § | Sección | ¿Se edita? | Qué hace |
|---|---|---|---|
| 1 | ⚙️ Entorno | No | Monta Drive, copia la librería a disco local, la instala |
| 1b | 📁 Rutas de datos | Solo si mueve los insumos | `RUTA_DATOS` (raíz de los insumos en Drive) y `RUTA_TRABAJO` (disco local) |
| 2 | 🧰 Definiciones | No | Helpers de lectura, contratos, perfilado y reportes |
| **3** | **🎛️ Parámetros** | **Sí — solo esta** | Rutas, columnas, pesos, umbrales, universo |
| 4 | 🔍 Preflight | No | Valida el mapeo contra el encabezado real, en segundos |
| 5 | 📖 Perfilado | No | Primeras filas, calidad de identificadores, cobertura |
| 6 | 🚀 Ejecución | No | Smoke test → cruce con checkpoints → invariantes → exportes |
| 7 | 📊 Resultados | No | Control de calidad y resultado de negocio |
| 8 | 📤 Entregables | No | Parquet de trabajo y ubicación de todo |
| 9 | ⏱️ Tiempos | No | Desglose por fase y total de la sesión |

Cada celda pesada lleva `%%time`; la §9 arma el desglose completo.

## 📁 Dónde queda cada cosa

| Cosa | Ruta |
|---|---|
| Librería (ya descomprimida) | `/content/drive/MyDrive/ProColombia/rues_linker_pruebas` |
| Insumos | `RUTA_DATOS/RUES/…` y `RUTA_DATOS/DANE/…` (§1b; por defecto `.../0A. Datos`) |
| Caché reutilizable | `.../0A. Datos/RUES/_procesados/rues_linker/` |
| Resultados | `.../0A. Datos/RUES/_procesados/resultados_rues_linker/` (`golden.parquet`, `correlativa.parquet`, `metadatos_corrida.json`) |
| Temporales | `/content/_rues_linker_trabajo` (disco local, se pierde con la sesión) |

Los datos **nunca** quedan dentro de la carpeta de la librería: mezclarlos hace
que actualizar el código borre meses de proceso.

## ♻️ Qué se reutiliza

Leer y proyectar el histórico del RUES cuesta minutos. La primera corrida deja
un Parquet con **solo las columnas que el cruce usa**; de ahí en adelante esa
lectura son segundos (medido: 8,6 s → 0,1 s). La caché guarda aparte el tamaño
y la fecha del original:

- Si el archivo o el mapeo de columnas cambian → **se descarta sola**.
- Si el archivo no está a mano (Drive sin montar) → **se usa igual**, avisando.
- `FORZAR_RELECTURA = True` la ignora.

Los **checkpoints del cruce** viven aparte en `RUTA_TRABAJO`: si la sesión se
cae, volver a ejecutar la §6 retoma donde iba.

## ⚠️ Tres cosas propias de estos datos

1. **El NIT viene en dos formatos.** El RUES trae la base de 9 dígitos; el DANE
   trae 10 (base + dígito de verificación). El pipeline canoniza ambos — siempre
   que el DV declarado sea correcto. La §7 cuantifica cuántos llegan mal.
2. **La base del DANE es transaccional**: ~895.000 filas para ~19.400
   exportadores. `COLAPSAR_DUPLICADOS = True` procesa un representante por
   combinación idéntica y luego restituye **todas** las filas originales.
3. **Centinelas reales**: `-1` en el NIT y `NO DEFINIDO` en la razón social se
   declaran nulos. Sin eso, "NO DEFINIDO" se comporta como una empresa.

## 📏 Mediciones de referencia

**Su corrida del universo completo** (Colab Free, 27-ago-2026, este mismo
notebook) — es la vara real de este montaje:

| Métrica | Valor |
|---|---|
| Filas de entrada | 5.264.104 (RUES 4.369.002 + DANE 895.102) |
| Entidades | 4.369.243 |
| Tiempo total | 42,8 min (fase de cruce: 42,1 min) |
| Pico de RAM | 7,97 GB de 12,7 GB disponibles |
| Exportaciones enlazadas | 871.681 de 895.102 (**97,4 %**) |
| Entidades en ambas fuentes | 18.825 |
| Grupos con dos NIT válidos distintos | **0** |

**Datos de prueba** — RUES 57.186 × Exportaciones 895.102 filas, 2 vCPU:
952.288 filas restituidas, 74.242 entidades, 93,3 s, 956 MiB de pico.

## 🎯 Calidad medida

Primera versión calibrada contra **ground truth etiquetado**
(7.368 registros, 2.000 grupos, ruido controlado hasta EXTREME):

| Perfil | Precisión | Recall | F1 | B-cubed F1 |
|---|---|---|---|---|
| `produccion_estandar` (este caso) | 1,000 | 0,364 | 0,534 | 0,749 |
| `fuentes_ruidosas` (CRM / captura manual) | 1,000 | 0,771 | **0,871** | **0,924** |

Léalo bien antes de cambiar el perfil: el recall bajo del estándar **no es un
defecto aquí**. Ese ground truth castiga deliberadamente el identificador
corrupto, y el RUES es la fuente oficial del NIT — dos NIT válidos distintos
SON dos empresas. `fuentes_ruidosas` gana recall relajando eso, y sobre el
RUES real fusiona 14 grupos de más. Para RUES × DANE: `produccion_estandar`.

---
## ⚙️ §1 · Entorno

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  ⚙️ CELDA A — ENTORNO  (ejecutar UNA vez por sesion; no requiere edicion)
# ══════════════════════════════════════════════════════════════════════════
# Instala la libreria desde la RUEDA (.whl) que viene en el zip, no desde el
# arbol de 250+ archivos. Por que importa: copiar cientos de archivos pequeños
# desde Drive (FUSE) puede quedar INCOMPLETO sin lanzar error —Drive sincroniza
# de forma asincrona— y entonces pip construye un paquete al que le faltan
# modulos. El sintoma es "No module named 'record_linkage.config'" DESPUES de
# que pip dijo que instalo bien. Una rueda es UN archivo: o esta completa o
# falla ruidosamente. Ademas evita el paso de construccion (mas rapido).
from pathlib import Path
import importlib, shutil, subprocess, sys, zipfile

RUTA_LIBRERIA = Path("/content/drive/MyDrive/ProColombia/rues_linker_pruebas")
VERSION_ESPERADA = "0.22.4"

# ── 1. Drive ──────────────────────────────────────────────────────────────
if Path("/content").is_dir() and not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

if not RUTA_LIBRERIA.is_dir():
    raise FileNotFoundError(
        f"No existe {RUTA_LIBRERIA}.\n"
        "Descomprima el zip en Drive y apunte RUTA_LIBRERIA a la carpeta que "
        "contiene pyproject.toml y dist/ (en el zip se llama rues_linker_pruebas)."
    )

# ── 2. Buscar la rueda, en este orden ─────────────────────────────────────
_candidatas = [
    *sorted(RUTA_LIBRERIA.glob("dist/*.whl")),      # <- donde la deja el zip
    *sorted(RUTA_LIBRERIA.glob("*.whl")),
    *sorted(RUTA_LIBRERIA.parent.glob("*.whl")),
]
_rueda = next((r for r in _candidatas if VERSION_ESPERADA in r.name), None)
if _rueda is None and _candidatas:
    _rueda = _candidatas[0]
    print(f"⚠️ Ninguna rueda dice {VERSION_ESPERADA}; se usa {_rueda.name}")

# ── 3. Instalar ───────────────────────────────────────────────────────────
# NO se usa `pip install -e`: un editable deja un .pth que solo se lee AL
# ARRANCAR el interprete, asi que en un kernel ya en marcha pip devuelve 0 y
# `import record_linkage` falla igual.
def _pip(*args: str) -> subprocess.CompletedProcess:
    return subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *args],
        capture_output=True, text=True,
    )

if _rueda is not None:
    # Una rueda truncada por sincronizacion a medias se detecta aqui, no a
    # mitad de una corrida de 40 minutos.
    if not zipfile.is_zipfile(_rueda):
        raise RuntimeError(
            f"La rueda {_rueda.name} no es un archivo valido ({_rueda.stat().st_size:,} bytes).\n"
            "Drive no termino de sincronizarla. Espere a que el icono de "
            "sincronizacion desaparezca y vuelva a ejecutar esta celda."
        )
    _local = Path("/content") / _rueda.name if Path("/content").is_dir() else _rueda
    if _local != _rueda:
        shutil.copyfile(_rueda, _local)
    _origen_desc = f"rueda {_local.name}"
    print(f"Instalando {_origen_desc} ({_local.stat().st_size / 1024:.0f} KB) ...")
    _r = _pip(str(_local))                                   # paquete + dependencias
    if _r.returncode == 0:
        # Segunda pasada sin dependencias: garantiza que los archivos de ESTE
        # paquete queden completos aunque hubiera una instalacion previa rota.
        _r = _pip("--force-reinstall", "--no-deps", str(_local))
else:
    # Sin rueda: se instala desde el fuente, pero VERIFICANDO primero que el
    # arbol este completo. Instalar un fuente incompleto es exactamente el
    # fallo que esta celda existe para evitar.
    _paquete = RUTA_LIBRERIA / "src" / "record_linkage"
    _ESPERADOS = [
        "__init__.py", "api.py", "config/__init__.py", "config/paths.py",
        "config/profiles.py", "config/settings.py", "engine/__init__.py",
        "flujo/__init__.py", "flujo/cruce.py", "golden/__init__.py",
        "ingestion/__init__.py", "pipeline/__init__.py", "processing/__init__.py",
        "processing/text.py", "processing/_constants.py", "reporting/__init__.py",
        "utils/__init__.py",
    ]
    _faltan = [f for f in _ESPERADOS if not (_paquete / f).is_file()]
    if _faltan:
        raise FileNotFoundError(
            "El arbol de la libreria en Drive esta INCOMPLETO. Faltan:\n"
            + "".join(f"  - src/record_linkage/{f}\n" for f in _faltan)
            + "\nCausa habitual: Drive no termino de sincronizar el zip "
              "descomprimido. Que hacer: espere a que Drive termine, o (mejor) "
              "use el zip que trae dist/*.whl y deje que esta celda instale la "
              "rueda, que es UN archivo y no puede quedar a medias."
        )
    _local = Path("/content/rues_linker_lib") if Path("/content").is_dir() else RUTA_LIBRERIA
    if _local != RUTA_LIBRERIA:
        shutil.rmtree(_local, ignore_errors=True)
        shutil.copytree(
            RUTA_LIBRERIA, _local,
            ignore=shutil.ignore_patterns(
                ".git", "__pycache__", "*.egg-info", "build", "dist",
                ".pytest_cache", ".ruff_cache", ".mypy_cache",
            ),
        )
    _origen_desc = f"fuente {_local.name}"
    print(f"Instalando {_origen_desc} (no se hallo rueda en dist/) ...")
    _r = _pip(str(_local))
    if _r.returncode == 0:
        _r = _pip("--force-reinstall", "--no-deps", str(_local))

if _r.returncode != 0:
    print(_r.stdout[-3000:]); print(_r.stderr[-3000:])
    raise RuntimeError("pip install fallo; el detalle esta arriba.")

# ── 4. Descartar cualquier modulo viejo y cargar limpio ───────────────────
# Si el kernel ya habia importado otra version, `reload` no basta: los
# submodulos siguen apuntando al codigo viejo.
for _m in [m for m in list(sys.modules) if m == "record_linkage" or m.startswith("record_linkage.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

# ── 5. Verificar que el paquete quedo COMPLETO ────────────────────────────
# Un paquete a medias importa el __init__ y revienta en el primer submodulo.
# Aqui se comprueba explicitamente y se dice CUAL falta y DONDE se busco.
_SUBPAQUETES = (
    "config.paths", "config.profiles", "config.settings", "api",
    "engine.scorer", "engine.cannot_link", "flujo.cruce", "golden.generator",
    "ingestion.duckdb", "pipeline.orchestrator", "processing.text",
)
try:
    record_linkage = importlib.import_module("record_linkage")
except ModuleNotFoundError as exc:
    raise RuntimeError(
        f"El paquete quedo incompleto: falta {exc.name!r}.\n"
        f"Se instalo desde: {_origen_desc}\n"
        "Causa casi segura: la copia en Drive esta a medias. Reemplace la "
        "carpeta con el zip completo (debe traer dist/*.whl) y repita."
    ) from exc

_rotos = []
for _sub in _SUBPAQUETES:
    try:
        importlib.import_module(f"record_linkage.{_sub}")
    except Exception as exc:  # noqa: BLE001 - se reporta, no se oculta
        _rotos.append(f"{_sub}: {type(exc).__name__}: {exc}")
if _rotos:
    raise RuntimeError(
        "El paquete se importo pero tiene modulos rotos o ausentes:\n"
        + "".join(f"  - {r}\n" for r in _rotos)
        + f"\nInstalado desde: {_origen_desc}\n"
          f"Ubicacion: {Path(record_linkage.__file__).parent}"
    )

# ── 6. La version cargada es la que usted cree ────────────────────────────
if record_linkage.__version__ != VERSION_ESPERADA:
    raise RuntimeError(
        f"Se cargo rues-linker {record_linkage.__version__} y este notebook "
        f"espera {VERSION_ESPERADA}.\n"
        f"Cargado desde: {Path(record_linkage.__file__).parent}\n"
        "Que hacer: confirme RUTA_LIBRERIA, luego Entorno de ejecucion > "
        "Reiniciar sesion y ejecute esta celda de nuevo."
    )

print(f"✅ rues-linker {record_linkage.__version__} listo | Python {sys.version.split()[0]}")
print(f"   Origen: {_origen_desc}")
print(f"   Cargado desde: {Path(record_linkage.__file__).parent}")
print(f"   Verificados {len(_SUBPAQUETES)} subpaquetes: todos importan")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  ⚙️ §1b · RUTAS DE DATOS  (carpeta raiz de los insumos y disco de trabajo)
# ══════════════════════════════════════════════════════════════════════════
# RUTA_DATOS es la carpeta de Drive que contiene RUES/ y DANE/; los nombres de
# archivo se declaran en la §3. RUTA_TRABAJO es disco LOCAL de la VM: los
# checkpoints SQLite sobre Drive (FUSE) son justamente lo que rompe las
# corridas largas. tests/test_notebooks_contrato.py sobreescribe ambas para
# correr este notebook sobre datos sinteticos.
from pathlib import Path

RUTA_DATOS = Path("/content/drive/MyDrive/ProColombia/0A. Datos")
RUTA_TRABAJO = Path("/content/_rues_linker_trabajo")  # disco LOCAL, nunca Drive

---
## 🧰 §2 · Definiciones

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  🧰 §2 · DEFINICIONES  (ejecutar una vez; no requiere edicion)
# ══════════════════════════════════════════════════════════════════════════
# Todo lo pesado (preflight, cache anti-Drive, smoke test, invariantes,
# exportes, metadatos, cronometro) vive en `record_linkage.flujo`, que se
# entrega con pruebas. Aqui solo hay lo que traduce SUS parametros a
# contratos de fuente, y utilidades de lectura y reporte.
import gc
import time
import warnings
import zipfile
from pathlib import Path

# Silenciar TODO esconde justo lo que hay que ver (dtypes inferidos mal,
# columnas duplicadas, deprecaciones que cambian resultados). Solo se acotan
# las categorias ruidosas y conocidas de pandas/pyarrow al leer archivos
# grandes; cualquier otra advertencia llega intacta.
warnings.filterwarnings("ignore", category=FutureWarning, module=r"pandas\..*")
warnings.filterwarnings("ignore", message=".*Passing a BlockManager.*")
warnings.filterwarnings("ignore", message=".*swapaxes.*")

import pandas as pd

from record_linkage import (
    ColumnType,
    Compression,
    IdentifierFormat,
    InputFormat,
    SourceSpec,
    sugerir_esquema,
)
from record_linkage.golden.columnas_finales import COLUMNAS_FINALES
from record_linkage.flujo import (
    ConfigCruce,
    diagnosticar_identificadores,
    ejecutar_cruce,
    exportar_sin_pareja,
    pares_enlazados,
)
from record_linkage.ingestion import DuckDBIngestionSettings, resumir_universo
from record_linkage.flujo.insumos import (
    cargar_fuente_con_cache,
    filas_en_cache,
    ruta_en_cache,
)

pd.options.display.max_columns = None
pd.set_option("display.max_colwidth", 60)

#: Cronometro de la sesion completa (se reporta en la §8).
_T0_SESION = time.time()

#: Columnas canonicas que el motor entiende. NIT y RAZON_SOCIAL son
#: obligatorias; el resto son evidencia adicional opcional.
CANONICAS = ("NIT", "RAZON_SOCIAL", "CIUDAD", "DEPARTAMENTO", "TELEFONO", "EMAIL", "DIRECCION")

#: Alias conocidos por canonica. Se usan cuando usted escribe "auto" en la §3.
#: Cada entrega del RUES ha usado nombres distintos (la de 42 columnas dice
#: CODIGO_MUNICIPIO_COMERCIAL; el historico consolidado de 56 dice MUNICIPIO):
#: en vez de adivinar, se busca en esta lista contra el encabezado real. Una
#: entrega nueva se soporta agregando un alias aqui, sin tocar nada mas.
ALIAS = {
    "NIT": ("NUMERO_IDENTIFICACION", "NIT", "LLAVE_NIT", "IDENTIFICACION",
            "NUM_IDENTIFICACION", "NIT_EXPORTADOR", "NIT EXPORTADOR", "DOCUMENTO"),
    "RAZON_SOCIAL": ("RAZON_SOCIAL", "RAZON SOCIAL", "NOMBRE", "NOMBRE_EMPRESA",
                     "NOMBRE_O_RAZON_SOCIAL", "EXPO_RAZON_SOCIAL"),
    "CIUDAD": ("MUNICIPIO", "COD_MUNICIPIO", "CODIGO_MUNICIPIO_COMERCIAL",
               "COD_MUNICIPIO_COMERCIAL", "MUNICIPIO_COMERCIAL", "CIUDAD"),
    "DEPARTAMENTO": ("DEPARTAMENTO", "COD_DEPARTAMENTO", "DEPARTAMENTO_ORIGEN",
                     "DEPARTAMENTO ORIGEN", "DPTO", "DEPTO"),
    "TELEFONO": ("TELEFONO_COMERCIAL_1", "TELEFONO_COMERCIAL", "TELEFONO_1",
                 "TELEFONO", "TEL", "CELULAR"),
    "EMAIL": ("CORREO_ELECTRONICO_COMERCIAL", "CORREO_ELECTRONICO", "EMAIL",
              "CORREO", "E_MAIL"),
    "DIRECCION": ("DIRECCION_COMERCIAL", "DIRECCION", "DIRECCION_FISCAL"),
}


def _clave(nombre: str) -> str:
    """Normaliza un nombre de columna para comparar sin may/min, espacios ni tildes."""
    import unicodedata

    texto = unicodedata.normalize("NFKD", str(nombre).strip().upper())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto.replace(" ", "_").replace("-", "_").replace(".", "")


# ──────────────────────────────────────────────────────────────────────────
#  Lectura de encabezados: adaptarse a la entrega real, no adivinar
# ──────────────────────────────────────────────────────────────────────────
def leer_encabezado(ruta) -> tuple[list[str], str | None]:
    """Devuelve (columnas, separador) leyendo SOLO la primera linea.

    Permite validar el mapeo antes de tocar gigabytes. La codificacion NO se
    decide aqui: un encabezado ASCII decodifica bien en cualquiera y elegir con
    eso llevaria a leer mal el cuerpo. Esa deteccion la hace la libreria sobre
    una muestra grande con `encoding="auto"`.

    Args:
        ruta: archivo .zip, .csv, .txt o .parquet.

    Returns:
        Tupla (nombres de columna, separador detectado; None si es Parquet).
    """
    ruta = Path(ruta)
    if ruta.suffix.lower() in {".parquet", ".pq"}:
        import pyarrow.parquet as pq

        return list(pq.read_schema(ruta).names), None
    if ruta.suffix.lower() == ".zip":
        with zipfile.ZipFile(ruta) as z:
            miembro = max((i for i in z.infolist() if not i.is_dir()), key=lambda i: i.file_size)
            with z.open(miembro) as f:
                crudo = f.readline()
    else:
        with ruta.open("rb") as f:
            crudo = f.readline()
    linea = crudo.decode("utf-8-sig", errors="replace").rstrip("\r\n")
    separador = max((",", ";", "\t", "|"), key=linea.count)
    return [c.strip().strip('"') for c in linea.split(separador)], separador


def ver_columnas(ruta, por_fila: int = 4) -> list[str]:
    """Imprime TODAS las columnas del archivo, numeradas. Uselo para mapear.

    Con 56 columnas y nombres que cambian entre entregas, esto es mas rapido y
    mas seguro que abrir el archivo aparte.

    Args:
        ruta: archivo a inspeccionar.
        por_fila: cuantos nombres imprimir por linea.

    Returns:
        La lista de columnas, por si quiere manipularla.
    """
    columnas, separador = leer_encabezado(ruta)
    print(f"📄 {Path(ruta).name} · {len(columnas)} columnas · separador={separador!r}")
    for i in range(0, len(columnas), por_fila):
        print("   " + " | ".join(f"{j:>2}. {columnas[j]:<32}"
                                 for j in range(i, min(i + por_fila, len(columnas)))))
    return columnas


def construir_fuente(
    nombre: str,
    ruta,
    columnas: dict,
    *,
    nulos_por_columna: dict | None = None,
    encoding: str = "auto",
    separador: str | None = None,
) -> SourceSpec:
    """Traduce un mapeo `{canonica: columna_del_archivo}` a un contrato validado.

    Valida contra el encabezado REAL: si declara una columna que no existe,
    falla aqui —en segundos— y le dice cuales si estan, en vez de reventar
    dentro de una corrida de horas. Las canonicas con valor None se omiten.

    Args:
        nombre: nombre logico de la fuente (aparece en la correlativa como SRC).
        ruta: archivo de entrada.
        columnas: `{canonica: nombre_en_el_archivo}`. NIT y RAZON_SOCIAL
            obligatorias; el resto opcionales.
        nulos_por_columna: centinelas propios del archivo, p. ej.
            `{"NIT": ("-1", "0"), "RAZON_SOCIAL": ("NO DEFINIDO",)}`.
        encoding: "auto" (recomendado) o una codificacion explicita.
        separador: fuerza el delimitador; None lo detecta del encabezado.

    Returns:
        SourceSpec listo para el flujo.

    Raises:
        ValueError: si falta una canonica obligatoria o si alguna columna
            declarada no existe en el archivo.
    """
    ruta = Path(ruta)
    disponibles, sep_detectado = leer_encabezado(ruta)
    # El indice se normaliza con la MISMA funcion que la busqueda: si una
    # normaliza espacios a guion bajo y la otra no, "Nit Exportador" deja de
    # encontrarse. (Defecto real, atrapado al ejecutar el notebook completo.)
    indice = {_clave(c): c for c in disponibles}

    declaradas = {k: v for k, v in columnas.items() if v}
    resueltas, omitidas, faltan_obligatorias = {}, [], []
    for canonica, pedido in declaradas.items():
        candidatos = ALIAS.get(canonica, ()) if str(pedido).lower() == "auto" else (pedido,)
        encontrada = next((indice[_clave(c)] for c in candidatos if _clave(c) in indice), None)
        if encontrada is not None:
            resueltas[canonica] = encontrada
        elif canonica in ("NIT", "RAZON_SOCIAL"):
            faltan_obligatorias.append(f"{canonica} -> {pedido!r}")
        else:
            omitidas.append((canonica, pedido))

    # Solo lo OBLIGATORIO tumba la corrida. Que falte un telefono no puede
    # costar una hora de proceso: se avisa, se omite y se sigue.
    if faltan_obligatorias:
        raise ValueError(
            f"[{nombre}] no encuentro columna(s) OBLIGATORIA(s) en {ruta.name}:\n  "
            + "\n  ".join(faltan_obligatorias)
            + f"\n\nColumnas disponibles ({len(disponibles)}):\n  "
            + "\n  ".join(", ".join(disponibles[i:i + 5]) for i in range(0, len(disponibles), 5))
            + "\n\nCorrija el mapeo en la §3, o escriba \"auto\" para que se resuelva "
              "por la tabla ALIAS. `ver_columnas(<ruta>)` las lista todas."
        )
    for canonica, pedido in omitidas:
        if str(pedido).lower() == "auto":
            intentados = ", ".join(ALIAS.get(canonica, ())[:4])
            motivo = f"ningun alias conocido existe aqui ({intentados}...)"
        else:
            motivo = f"no existe la columna {pedido!r}"
        raiz = _clave(canonica)[:4]
        parecidas = [c for c in disponibles if raiz in _clave(c)][:3]
        print(f"  ⚠️  {nombre}: se omite {canonica} — {motivo}"
              + (f" · parecidas en el archivo: {parecidas}" if parecidas else ""))

    es_parquet = ruta.suffix.lower() in {".parquet", ".pq"}
    obligatorias = {k: resueltas[k] for k in ("NIT", "RAZON_SOCIAL")}
    opcionales = {k: v for k, v in resueltas.items() if k not in obligatorias}

    print(f"  ✓ {nombre:<16} {len(disponibles):>3} columnas | " + " · ".join(
        f"{k}='{v}'" for k, v in resueltas.items()))

    return SourceSpec(
        name=nombre,
        path=ruta,
        format=InputFormat.PARQUET if es_parquet else InputFormat.AUTO,
        compression=(
            Compression.NONE if es_parquet
            else (Compression.ZIP if ruta.suffix.lower() == ".zip" else Compression.AUTO)
        ),
        delimiter=None if es_parquet else (separador or sep_detectado),
        encoding="utf-8" if es_parquet else encoding,
        column_mapping=obligatorias,
        optional_column_mapping=opcionales,
        column_types={"NIT": ColumnType.IDENTIFIER, "RAZON_SOCIAL": ColumnType.STRING},
        identifier_formats={"NIT": IdentifierFormat(mode="digits", min_length=6, max_length=12)},
        null_values=nulos_por_columna or {},
    )


# ──────────────────────────────────────────────────────────────────────────
#  Preflight e informes
# ──────────────────────────────────────────────────────────────────────────
def estado_cache(fuentes, dir_procesados) -> None:
    """Antes de correr: que se reutiliza y que se va a leer entero."""
    print("Cache de insumos:")
    for spec in fuentes:
        ruta = ruta_en_cache(spec, dir_procesados)
        if ruta.is_file():
            print(f"  ♻️  {spec.name:<16} reutilizable ({ruta.stat().st_size / 1024**2:,.0f} MB)")
        else:
            print(f"  📖 {spec.name:<16} se leera completa (aun sin cache)")


def perfilar(fuentes, dir_procesados=None, filas: int = 3, muestra: int = 1_000_000) -> None:
    """Muestra primeras filas y diagnostico de identificadores por fuente.

    Con la cache tibia cuesta segundos, y es lo que evita descubrir un mapeo
    equivocado despues de una hora de proceso.

    Args:
        muestra: tope de filas para las estadisticas por columna. A 4,4 M de
            filas el conteo exacto de valores distintos cuesta minutos y no
            cambia la decision; por encima del tope se estima sobre una
            muestra y se marca con "~".
    """
    for spec in fuentes:
        datos, _reporte, cacheado = cargar_fuente_con_cache(spec, dir_procesados)
        print(f"\n{'─' * 70}\n📄 {spec.name}{'  (de cache)' if cacheado else ''}"
              f"  ·  {len(datos):,} filas × {len(datos.columns)} columnas")
        display(datos.head(filas))
        print(diagnosticar_identificadores(datos["NIT"]).resumen())
        aproximado = len(datos) > muestra
        base = datos.head(muestra) if aproximado else datos
        marca = "~" if aproximado else " "
        for col in datos.columns:
            if col in ("NIT", "RAZON_SOCIAL"):
                continue
            vacios = base[col].isna().sum() + (base[col].astype("string").fillna("") == "").sum()
            print(f"    {col:<16} {marca}{base[col].nunique():>8,} valores distintos"
                  f"  ·  {vacios / max(len(base), 1):>6.1%} vacios"
                  + (f"  (muestra de {muestra:,})" if aproximado else ""))
        del datos, base
        gc.collect()


def liberar(*variables) -> None:
    """Libera memoria explicitamente y reporta la RAM disponible."""
    for variable in variables:
        del variable
    gc.collect()
    try:
        import psutil

        ram = psutil.virtual_memory()
        print(f"🧹 RAM libre: {ram.available / 1024**3:.1f} GB de {ram.total / 1024**3:.1f} GB "
              f"({ram.percent:.0f}% usado)")
    except ImportError:
        print("🧹 gc.collect() ejecutado")


print("✅ Definiciones cargadas. Edite unicamente la §3 (PARAMETROS).")

---
## 🎛️ §3 · Parámetros

**Es la única celda que usted edita.** Todo lo configurable vive aquí: rutas, qué columnas usar de cada fuente, qué variables pesan en la decisión, los umbrales del motor y el tamaño del universo.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  🎛️ §3 · PARAMETROS  —  LA UNICA CELDA QUE USTED EDITA
# ══════════════════════════════════════════════════════════════════════════

# ── 3.1 ¿Que quiere hacer? ────────────────────────────────────────────────
MODO = "linkage"          # "linkage" = cruzar 2+ bases | "dedupe" = limpiar UNA

# ── 3.2 Rutas ─────────────────────────────────────────────────────────────
# Relativas a RUTA_DATOS (§1b): si mueve la carpeta de insumos, cambie solo esa.
RUTA_RUES = RUTA_DATOS / "RUES" / "2026_06_30_Historico_Limpio.zip"
RUTA_EXPORTACIONES = (
    RUTA_DATOS / "DANE" / "2026-08-21 Base_Exportaciones_Colombianas_2021-2026 (Junio).zip"
)
# Si otro proceso ya dejo el RUES canonico en Parquet, apuntelo aqui y se
# evita leer el ZIP completo. None = usar el ZIP original.
RUTA_RUES_PREPARADO = None

# Estas dos se derivan de RUTA_RUES: si mueve los insumos, se mueven solas.
# RUTA_TRABAJO (disco local) se fija en la §1b.
RUTA_PROCESADOS = RUTA_RUES.parent / "_procesados" / "rues_linker"   # cache reutilizable
RUTA_SALIDA     = RUTA_RUES.parent / "_procesados" / "resultados_rues_linker"

# ── 3.3 Que columnas usar de cada fuente ──────────────────────────────────
# `{canonica: nombre en el archivo}`. NIT y RAZON_SOCIAL son obligatorias.
# Las demas son EVIDENCIA ADICIONAL: agregue o quite libremente. Poner None
# desactiva esa canonica. Si declara una columna que no existe, la §4 se lo
# dice de una con la lista de las que si estan.
# "auto" resuelve el nombre real por la tabla ALIAS de la §2 — util porque
# cada entrega del RUES los cambia (la de 42 columnas dice
# CODIGO_MUNICIPIO_COMERCIAL; el historico consolidado de 56 dice MUNICIPIO).
# Una canonica OPCIONAL que no aparezca se omite con aviso; solo NIT y
# RAZON_SOCIAL detienen la corrida. Use `ver_columnas(RUTA_RUES)` para verlas.
COLUMNAS_RUES = {
    "NIT":          "auto",
    "RAZON_SOCIAL": "auto",
    "CIUDAD":       "auto",
    "DEPARTAMENTO": "auto",
    "TELEFONO":     "auto",
    "EMAIL":        "auto",
}
COLUMNAS_EXPORTACIONES = {
    "NIT":          "Nit Exportador",
    "RAZON_SOCIAL": "Razon Social",
    "DEPARTAMENTO": "Departamento Origen",
}

# Centinelas propios de cada archivo: valores que PARECEN dato pero no lo son.
NULOS_RUES = {"NIT": ("0", "00", "0000000000000"), "RAZON_SOCIAL": ("SIN DATO", "NO DEFINIDO")}
# Centinelas REALES de estas entregas. Declararlos los vuelve "nulo conocido"
# en vez de "identificador invalido sin explicar": en su corrida del universo
# completo fueron 2.747 filas con NIT "00" y 56 del RUES con ceros.
NULOS_EXPORTACIONES = {"NIT": ("-1", "0", "00"), "RAZON_SOCIAL": ("NO DEFINIDO",)}

# ── 3.4 Que columnas PESAN en la decision, ademas de NIT y nombre ─────────
# Solo tienen efecto las columnas presentes en TODAS las fuentes del cruce.
# type: exact_or_zero | categorical | token_set_ratio | *_signed (los
# firmados restan cuando hay evidencia en contra, no solo suman cuando cuadra).
# ⚠️ Cuidado con el DEPARTAMENTO en este cruce concreto: en el RUES es el
# DOMICILIO de la empresa y en el DANE es el departamento de ORIGEN del
# despacho. Una empresa de Bogota que exporta por Atlantico NO es un error,
# pero un comparador "firmado" lo castigaria como evidencia en contra. Si lo
# activa, uselo SIN firmar ("categorical"): suma cuando coincide y no resta
# cuando no. Con telefono o correo —que si significan lo mismo en ambas
# fuentes— el firmado si es apropiado.
VARIABLES_EXTRA = [
    # {"column": "DEPARTAMENTO", "weight": 0.08, "type": "categorical"},
    # {"column": "TELEFONO",     "weight": 0.10, "type": "exact_signed"},
]

# ── 3.5 Como comparar (motor) ─────────────────────────────────────────────
PERFIL = "produccion_estandar"
# prueba_rapida · produccion_estandar · produccion_exhaustiva ·
# alta_precision · deduplicacion_simple · produccion_calibrada ·
# fuentes_ruidosas  <- v0.17.0
#
# ¿Cual usar aqui? Para RUES x DANE: produccion_estandar. El RUES es la
# fuente oficial del identificador: si dos NIT con digito de verificacion
# valido difieren, son dos empresas y punto. `fuentes_ruidosas` es para
# bases CAPTURADAS A MANO (CRM, formularios, planillas) donde el NIT trae
# errores de tecleo; medido contra ground truth etiquetado sube el F1 de
# 0,53 a 0,87, pero sobre el RUES real fusiona 14 grupos de mas.

AJUSTES = {
    # Bloqueo: cuantos pares se llegan a evaluar.
    "lsh_permutations": 252,   # + permutaciones = firma mas fiel, mas costo
    "lsh_threshold":    0.55,  # + alto = menos candidatos, mas rapido, menos recall
    "lsh_ngram":        3,
    # Decision: cuando dos registros son la misma entidad.
    "score_threshold":     0.45,  # + alto = mas precision, menos recall
    "min_name_similarity": 0.30,
    "max_nit_distance":    3,     # tolerancia de typo SOLO si el DV no valida
    # Normalizacion del nombre.
    "cleaning_mode":    "BALANCEADO",  # AGRESIVO | BALANCEADO | CONSERVADOR
    "remove_top_words": 0,             # quita las N palabras mas frecuentes del corpus
    # Salvaguardas de precision (v0.14.0). Cambiarlas exige una razon.
    "veto_nit_base_distinto":   True,
    "cannot_link_identificador": True,
    # v0.17.0 — el identificador como EVIDENCIA GRADUADA. Con 0 (default) dos
    # NIT base validos distintos jamas se unen: la semantica institucional
    # correcta para el RUES. Con d>0 (1-3), dos bases a distancia d de
    # digitacion (typo, digito de mas/menos, transposicion) Y con nombre
    # cohesivo se tratan como el MISMO identificador mal capturado. Suba esto
    # solo si su fuente captura el NIT a mano; sobre RUES x DANE dejelo en 0.
    "tolerancia_digitacion_identificador":  0,
    "similitud_nombre_rescate_identificador": 0.90,
}
PERFIL_MULTICAMPO = None   # None | "colombia" | "international"

# ── 3.6 Universo y ejecucion ──────────────────────────────────────────────
# None = universo COMPLETO. Para el primer montaje de una fuente nueva use
# {"RUES": 300_000}: valida todo en minutos. Ojo: con muestra el porcentaje de
# enlace es bajo POR CONSTRUCCION (entra solo una fraccion del RUES).
MUESTRA = None
CONFIABLES = {"RUES"}         # fuentes con identificador verificado

# ── 3.7 Motor: cuanta RAM se compromete ───────────────────────────────────
# "auto"    : RECOMENDADO. Mide el universo en la §4 y elige solo. Por debajo
#             del umbral usa pandas (mas comodo, devuelve DataFrames); por
#             encima cambia a disco. Asi la corrida no depende de que usted
#             acierte el modo: por grande que sea la base, termina.
# "pandas"  : el universo se procesa en memoria. Devuelve DataFrames.
# "duckdb"  : ingesta, compactacion y publicacion EN DISCO. El pico de RAM
#             deja de crecer con el numero de filas fisicas; la salida son
#             Parquet consultables. La §7 y la §8 funcionan IGUAL en los dos
#             modos: la libreria responde las mismas preguntas por SQL.
MOTOR = "auto"
# Umbral del modo "auto". 1,5 M de filas fisicas deja ~3x de margen sobre lo
# medido en Colab Free antes de que el camino pandas apriete la RAM.
UMBRAL_FILAS_DISCO = 1_500_000
LIMITE_RAM_DUCKDB = "1GB"     # presupuesto del buffer de DuckDB, no del proceso
HILOS_DUCKDB = 2
FILAS_SMOKE = 5_000           # ensayo previo; 0 lo desactiva
COLAPSAR_DUPLICADOS = True    # base transaccional: ~12x menos trabajo
FORZAR_RELECTURA = False      # True ignora la cache de insumos
REUSAR_CHECKPOINTS = True     # False rehace el cruce desde cero
EXPORTAR_EXCEL = False        # ~1M de filas no caben en Excel

# ══════════════════ FIN DE LOS PARAMETROS ══════════════════
# De aqui en adelante no hay nada que editar.

_DECLARADAS = [
    ("RUES", RUTA_RUES_PREPARADO or RUTA_RUES, COLUMNAS_RUES, NULOS_RUES),
    ("EXPORTACIONES", RUTA_EXPORTACIONES, COLUMNAS_EXPORTACIONES, NULOS_EXPORTACIONES),
]
if MODO == "dedupe":
    _DECLARADAS = _DECLARADAS[:1]
    CONFIABLES = set()

print(f"Modo: {MODO.upper()} · {len(_DECLARADAS)} fuente(s) · perfil '{PERFIL}'"
      f"{' + multicampo ' + str(PERFIL_MULTICAMPO) if PERFIL_MULTICAMPO else ''}")
print(f"Universo: {'completo' if not MUESTRA else 'MUESTRA ' + str(MUESTRA)}")

---
### 🔎 ¿No sabe cómo se llaman las columnas?

Ejecute `ver_columnas(RUTA_RUES)` en una celda vacía: las lista todas, numeradas. Cada entrega del RUES cambia los nombres — la de 42 columnas dice `CODIGO_MUNICIPIO_COMERCIAL`, el histórico consolidado de 56 dice `MUNICIPIO`—, por eso la §3 viene con `"auto"`: resuelve el nombre real contra la tabla `ALIAS`. Si una canónica **opcional** no aparece, se omite con aviso; solo `NIT` y `RAZON_SOCIAL` detienen la corrida.

---
## 🔍 §4 · Preflight

Valida el mapeo contra el encabezado **real** de cada archivo, resuelve los `"auto"`, avisa qué se omite y estima cuánto va a costar la corrida antes de comprometer la sesión.

In [ ]:
%%time
# ══════════════════════════════════════════════════════════════════════════
#  🔍 §4 · PREFLIGHT  —  validar el montaje ANTES de gastar tiempo
# ══════════════════════════════════════════════════════════════════════════
# Construir los contratos ya valida contra el encabezado real: si declaro una
# columna que no existe, esto falla en segundos y le dice cuales si estan.
print("Contratos de fuente")
FUENTES = [
    construir_fuente(nombre, ruta, cols, nulos_por_columna=nulos)
    for nombre, ruta, cols, nulos in _DECLARADAS
]

print()
estado_cache(FUENTES, RUTA_PROCESADOS)

# Las variables extra solo sirven si la columna existe en TODAS las fuentes.
_canonicas_por_fuente = [
    set(f.column_mapping) | set(f.optional_column_mapping) for f in FUENTES
]
_comunes = set.intersection(*_canonicas_por_fuente) if _canonicas_por_fuente else set()
_huerfanas = [v["column"] for v in VARIABLES_EXTRA if v["column"] not in _comunes]
if _huerfanas:
    raise ValueError(
        f"VARIABLES_EXTRA menciona {_huerfanas}, que no esta(n) en todas las fuentes.\n"
        f"Columnas comunes disponibles: {sorted(_comunes - {'NIT', 'RAZON_SOCIAL'}) or '(ninguna)'}\n"
        f"Mapee esa columna en cada fuente en la §3, o quitela de VARIABLES_EXTRA."
    )
print(f"\nColumnas comunes a todas las fuentes: {sorted(_comunes)}")
print(f"Pesan en el score: NIT, RAZON_SOCIAL"
      + (f" + {[v['column'] for v in VARIABLES_EXTRA]}" if VARIABLES_EXTRA else ""))

# ── Dimensionado del universo ─────────────────────────────────────────────
# La decision "RAM o disco" hay que tomarla AQUI, antes de leer nada: tomada
# mal, la sesion muere por OOM a los 40 minutos con todo el trabajo perdido.
# Hasta 0.17.2 esta celda solo sabia el tamaño si YA existia cache de una
# corrida previa -es decir, justo en la primera corrida, la peligrosa, no
# sabia nada y caia a pandas-. Ahora:
#   · si hay cache, el conteo sale del pie del Parquet (exacto y gratis);
#   · si no la hay, se estima leyendo 2 MiB por fuente (8 ventanas repartidas
#     por todo el archivo). Contra las bases reales el error fue -0,5 % (RUES)
#     y -1,7 % (Exportaciones), en ~1 s.
_exactas = {}
for spec in FUENTES:
    _n = filas_en_cache(spec, RUTA_PROCESADOS)
    if _n is not None:
        _exactas[spec.name] = _n
_universo = resumir_universo(FUENTES, exactas=_exactas)
print("\nTamaño de cada fuente")
for _e in _universo.por_fuente:
    _etq = "exacto" if _e.exacta else "estimado"
    _val = f"{_e.filas:,}" if _e.filas is not None else "desconocido"
    print(f"  · {_e.fuente:<16s} {_val:>12s} filas  ({_etq}, {_e.metodo})")
    if _e.filas is None:
        print(f"      {_e.detalle}")

_total = _universo.filas
_tope = (MUESTRA or {}).get("RUES")
if _total is not None and _tope:
    _por_fuente = {e.fuente: (e.filas or 0) for e in _universo.por_fuente}
    _total = min(_por_fuente.get("RUES", 0), _tope) + sum(
        v for k, v in _por_fuente.items() if k != "RUES")

# Estimacion de costo, para decidir ANTES de comprometer la sesion.
# Dos puntos MEDIDOS: 952.288 filas -> 1,6 min (2 vCPU, datos de prueba) y
# 5.265.102 filas -> 41,1 min (universo completo, 2 vCPU, verificado el
# 29-ago-2026 con 0.22.1). Entre ambos se interpola; por encima se extrapola
# con la pendiente del tramo grande. La ETA depende del disco de la VM:
# tomela como orden de magnitud, no como promesa.
_REF = [(952_288, 1.6), (5_265_102, 41.1)]

# ── Presupuesto de memoria ────────────────────────────────────────────────
# Son DOS presupuestos, y confundirlos fue el error de la version anterior de
# esta celda, que al elegir DuckDB afirmaba "el pico de RAM deja de crecer con
# el numero de filas". Es falso, y por creerlo se perdio una corrida entera:
#
#   INGESTA · leer, proyectar, normalizar y COLAPSAR duplicados exactos. Con
#             motor DuckDB esto ocurre EN DISCO: aqui el numero de filas
#             fisicas deja de gobernar la RAM.
#   MATCHER · L1 a L5 (limpieza, firmas, scoring, clustering, golden). Trabaja
#             sobre un DataFrame EN MEMORIA de los registros ya colapsados,
#             elija el motor que elija. Este es el presupuesto que manda.
#
# Medido el 29-ago-2026 con el universo real: 5.265.102 filas fisicas ->
# 4.389.407 registros colapsados, pico de 5,5 GB en 41 min. Es decir
# ~1,26 GB por millon de registros que entran al matcher.
_GB_POR_MILLON = 1.26      # medido, no estimado
_RAM_UTIL_COLAB = 11.0     # 12,7 GB menos lo que ocupa el runtime

if _total is not None:
    (_n1, _t1), (_n2, _t2) = _REF
    _eta = _t1 + (_total - _n1) * (_t2 - _t1) / (_n2 - _n1) if _total > _n1 else (
        _total / _n1 * _t1)
    # Cota SUPERIOR: supone que las fuentes no colapsan nada. RUES apenas
    # colapsa (4,37 M de filas -> 4,37 M de representantes), asi que para el
    # historico del RUES la cota superior es practicamente el valor real; una
    # base transaccional como exportaciones colapsa 46x y quedara muy por
    # debajo.
    _ram = _total / 1e6 * _GB_POR_MILLON
    print(f"\nUniverso a procesar: {_total:,} filas"
          + ("" if _universo.exacto else "  (estimado)"))
    print(f"  ETA de cruce      : ~{max(_eta, 0.2):.0f} min")
    print(f"  RAM del matcher   : hasta ~{_ram:.1f} GB "
          f"(~{_GB_POR_MILLON} GB por millon de registros tras colapsar)")
    if _ram < _RAM_UTIL_COLAB * 0.7:
        print(f"  Colab Free (~{_RAM_UTIL_COLAB:.0f} GB utiles): holgura suficiente.")
    elif _ram < _RAM_UTIL_COLAB:
        print(f"  Colab Free (~{_RAM_UTIL_COLAB:.0f} GB utiles): AJUSTADO. Si sus fuentes "
              f"colapsan poco,\n     recorte con MUESTRA o suba a una VM con mas RAM.")
    else:
        print(f"  ⚠️ Colab Free (~{_RAM_UTIL_COLAB:.0f} GB utiles) NO alcanza para "
              f"{_total:,} filas sin colapsar.\n"
              f"     El modo disco NO lo resuelve: cubre la ingesta, no el matcher.\n"
              f"     Opciones: recortar con MUESTRA, partir el universo, o mas RAM.")
else:
    print("\n⚠️ No se pudo dimensionar alguna fuente; se elegira el motor conservador.")

# ── Resolucion del motor ──────────────────────────────────────────────────
# Con MOTOR="auto" la decision se toma AQUI, con el tamaño real ya medido, no
# adivinando antes. El criterio es el numero de filas FISICAS: es lo que
# gobierna el pico de RAM del camino pandas.
if MOTOR == "auto":
    # Se compara contra el universo CON MARGEN: una cifra estimada se infla un
    # 25 % antes de decidir, para que el error de estimacion nunca empuje hacia
    # el motor que se cae. Y si no se pudo dimensionar, gana el disco: correr
    # mas lento es recuperable, quedarse sin RAM a los 40 minutos no lo es.
    _decision = _universo.filas_con_margen
    if _tope and _decision is not None and _total is not None:
        _decision = int(_total * (1.0 if _universo.exacto else 1.25))
    if _decision is None:
        MOTOR_EFECTIVO = "duckdb"
        print("\n⚙️ Motor: DUCKDB (disco) — no se pudo medir el universo, se toma el "
              "lado seguro.\n   Fije MOTOR = 'pandas' en la §3 si sabe que la base es chica.")
    elif _decision > UMBRAL_FILAS_DISCO:
        MOTOR_EFECTIVO = "duckdb"
        print(f"\n⚙️ Motor: DUCKDB (disco) — {_decision:,} filas (con margen) superan el "
              f"umbral de {UMBRAL_FILAS_DISCO:,}.\n"
              "   Cubre la INGESTA: proyeccion, normalizacion y colapso ocurren en disco,\n"
              "   y la salida queda en Parquet (la §7/§8 funcionan igual). NO cubre el\n"
              "   matcher: vea arriba el presupuesto de RAM, que es el que manda.")
    else:
        MOTOR_EFECTIVO = "pandas"
        print(f"\n⚙️ Motor: pandas — {_decision:,} filas (con margen) caben comodo en memoria.")
else:
    MOTOR_EFECTIVO = MOTOR
    print(f"\n⚙️ Motor: {MOTOR_EFECTIVO} (fijado a mano en la §3)")

---
## 📖 §5 · Perfilado de insumos

Opcional pero recomendado la primera vez sobre una fuente nueva. Con la caché tibia cuesta segundos.

In [ ]:
%%time
# ══════════════════════════════════════════════════════════════════════════
#  📖 §5 · PERFILADO DE INSUMOS  (opcional, pero recomendado la 1a vez)
# ══════════════════════════════════════════════════════════════════════════
# Primeras filas ya mapeadas al canonico, diagnostico de identificadores y
# cobertura de cada columna. Con la cache tibia cuesta segundos.
perfilar(FUENTES, RUTA_PROCESADOS)

---
## 🚀 §6 · Ejecución

Smoke test → cruce con checkpoints → verificación de invariantes → exportes y metadatos. Si la sesión se cae, volver a ejecutar retoma donde iba.

In [ ]:
%%time
# ══════════════════════════════════════════════════════════════════════════
#  🚀 §6 · EJECUCION
# ══════════════════════════════════════════════════════════════════════════
# La cache de insumos en Parquet (`dir_procesados`) pertenece al motor
# pandas: con DuckDB la proyeccion y la compactacion ocurren en disco dentro
# de `dir_trabajo`, asi que la libreria RECHAZA ese parametro en vez de
# aceptarlo sin efecto. Aqui se declara solo cuando aplica.
_ESPECIFICO_PANDAS = (
    {"dir_procesados": RUTA_PROCESADOS, "forzar_relectura": FORZAR_RELECTURA}
    if MOTOR_EFECTIVO == "pandas"
    else {}
)

CFG = ConfigCruce(
    fuentes=FUENTES,
    workspace=RUTA_SALIDA,
    dir_trabajo=RUTA_TRABAJO,
    **_ESPECIFICO_PANDAS,
    reusar_checkpoints=REUSAR_CHECKPOINTS,
    limite_filas={k: v for k, v in (MUESTRA or {}).items() if v} or None,
    confiables=CONFIABLES,
    variables_extra=VARIABLES_EXTRA or None,
    perfil=PERFIL,
    ajustes_perfil=AJUSTES,
    perfil_multicampo=PERFIL_MULTICAMPO,
    filas_smoke=FILAS_SMOKE,
    # DuckDB exige el colapso para conservar su contrato de expansion.
    colapsar_duplicados_exactos=COLAPSAR_DUPLICADOS or MOTOR_EFECTIVO == "duckdb",
    exportar_excel=EXPORTAR_EXCEL and MOTOR_EFECTIVO == "pandas",
    # El modo disco exige DuckDB y prohibe Excel: exportar un millon de filas
    # a xlsx rompe el presupuesto de memoria que este modo existe para cuidar.
    motor_ingesta=MOTOR_EFECTIVO,
    modo_resultado="disco" if MOTOR_EFECTIVO == "duckdb" else "dataframe",
    duckdb_settings=(
        DuckDBIngestionSettings(
            memory_limit=LIMITE_RAM_DUCKDB,
            threads=HILOS_DUCKDB,
            temp_directory=RUTA_TRABAJO / "duckdb_spill",
        )
        if MOTOR_EFECTIVO == "duckdb"
        else None
    ),
)

resultado = ejecutar_cruce(CFG)
print(resultado.resumen())

---
## 📊 §7 · Resultados y control de calidad

In [ ]:
%%time
# ══════════════════════════════════════════════════════════════════════════
#  📊 §7 · RESULTADOS Y CONTROL DE CALIDAD
# ══════════════════════════════════════════════════════════════════════════
# Esta seccion es IDENTICA en los dos motores: la libreria expone las mismas
# preguntas de calidad sobre un DataFrame (pandas) o sobre Parquet (duckdb),
# y en el segundo caso las resuelve en SQL sin traer la correlativa a RAM.

# 7.1 · Calidad de los identificadores que entraron.
for _fuente, _rep in resultado.identificadores_por_fuente().items():
    print(f"\n▸ {_fuente}")
    print(_rep.resumen())

# 7.2 · Verificacion dura: ningun grupo puede mezclar dos NIT validos distintos.
_conflictos = resultado.conflictos_identificador()
print(f"\n{'✅' if _conflictos == 0 else '❌'} Grupos con dos NIT validos distintos: "
      f"{_conflictos:,} (debe ser 0)")

# 7.2b · El entregable: qué identidad quedó para cada grupo.
# Sin NIT_FINAL y RAZON_SOCIAL_FINAL, la correlativa dice a qué grupo pertenece
# cada fila pero no qué NIT ni qué razón social se adoptaron para ese grupo, que
# es justo lo que se consume aguas abajo. La libreria lo garantiza desde 0.22.1
# y `identidad_adoptada` falla si falta, de modo que se ve aqui y no al abrir el
# parquet dias despues.
_faltan_finales = [
    _c for _c in COLUMNAS_FINALES if _c not in resultado.correlativa.columns
]
if _faltan_finales:
    raise RuntimeError(
        f"La correlativa salio sin {_faltan_finales}. No use este resultado: "
        f"revise el registro de la fase L5."
    )
print(f"\n✅ Contrato de salida completo: {list(COLUMNAS_FINALES)}")
print("Identidad adoptada por grupo (muestra):")
display(resultado.identidad_adoptada(limite=15))

# 7.3 · Tamano de los grupos.
# OJO con la metrica: en una base transaccional un grupo con miles de FILAS es
# normal (una empresa con miles de despachos). Lo que delata una sobre-fusion
# es la variedad de NOMBRES distintos dentro del grupo, no el conteo de filas.
_d = resultado.distribucion_grupos()
print(f"\nFilas por grupo   : 1 en {_d['grupos_1_fila']:,} grupos · "
      f"2-5 en {_d['grupos_2a5_filas']:,} · >5 en {_d['grupos_mas_5_filas']:,} · "
      f"maximo {_d['max_filas_por_grupo']:,}")
print(f"Nombres distintos : 1 en {_d['grupos_1_nombre']:,} grupos · "
      f"2-3 en {_d['grupos_2a3_nombres']:,} · >3 en {_d['grupos_mas_3_nombres']:,} · "
      f"maximo {_d['max_nombres_por_grupo']:,}")

_UMBRAL_VARIEDAD = 6   # a partir de aqui conviene mirar con ojo humano
_sospechosos = resultado.grupos_sospechosos(minimo_nombres=_UMBRAL_VARIEDAD, limite=20)
if len(_sospechosos):
    print(f"\n  ⚠️ Hay grupos con {_UMBRAL_VARIEDAD}+ nombres distintos. "
          f"Los mayores, para revision humana:")
    display(_sospechosos)
else:
    print(f"  ✅ Ningun grupo supera {_UMBRAL_VARIEDAD} nombres distintos.")

# 7.4 · El resultado de negocio (solo aplica a un cruce de 2+ fuentes).
if len(FUENTES) > 1:
    display(resultado.cruce_por_fuente())
    # Una fila por entidad enlazada, con el nombre de CADA fuente al lado: es
    # lo que permite auditar el match de un vistazo. Se pide acotado a la
    # libreria para no depender de tener la correlativa completa en memoria.
    _pares = pares_enlazados(resultado, limite=15)
    print(f"\nEntidades presentes en mas de una fuente: "
          f"{resultado.entidades_multifuente():,}")
    display(_pares)
else:
    print(f"\nEntidades: {resultado.metricas['entidades']:,} de "
          f"{resultado.metricas['filas_entrada']:,} filas "
          f"({1 - resultado.metricas['entidades'] / resultado.metricas['filas_entrada']:.2%} de reduccion)")
    display(resultado.grupos_sospechosos(minimo_nombres=2, limite=20))

---
## 📤 §8 · Entregables

In [ ]:
%%time
# ══════════════════════════════════════════════════════════════════════════
#  📤 §8 · ENTREGABLES
# ══════════════════════════════════════════════════════════════════════════
# `ejecutar_cruce` ya dejo golden.parquet, correlativa.parquet y
# metadatos_corrida.json en RUTA_SALIDA. Aqui se agregan las vistas de trabajo.
# Todo se resuelve en la libreria para que funcione igual con MOTOR="duckdb":
# en ese modo la seleccion ocurre DENTRO de DuckDB y la salida puede tener
# millones de filas sin pasar por la memoria del proceso.
print("Ya escritos por el flujo:")
for nombre, ruta in resultado.rutas.items():
    print(f"  {nombre:<22} {Path(ruta).name}")

if len(FUENTES) > 1:
    _principal = sorted(CONFIABLES)[0] if CONFIABLES else FUENTES[0].name
    _ruta, _n = exportar_sin_pareja(
        resultado,
        fuente_principal=_principal,
        destino=RUTA_SALIDA / f"sin_pareja_en_{_principal}.parquet",
    )
    print(f"\n📌 Sin pareja en {_principal}: {_n:,} → {_ruta.name}")

    _pares = pares_enlazados(resultado, limite=100_000)
    if len(_pares):
        _pares.to_parquet(RUTA_SALIDA / "entidades_cruzadas.parquet")
        print(f"📌 Entidades cruzadas: {len(_pares):,} → entidades_cruzadas.parquet")

print(f"\n📂 Todo en: {RUTA_SALIDA}")

---
## ⏱️ §9 · Tiempos y cierre

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  ⏱️ §9 · TIEMPOS Y CIERRE DE SESION
# ══════════════════════════════════════════════════════════════════════════
print(resultado.tiempos())

_m = resultado.metricas
print(f"\nFilas de entrada : {_m['filas_entrada']:>12,}")
print(f"Entidades        : {_m['entidades']:>12,}")
print(f"Reduccion        : {1 - _m['entidades'] / max(_m['filas_entrada'], 1):>11.2%}")
print(f"Por fuente       : {_m['por_fuente']}")
print(f"\nSesion completa (desde la §2): {(time.time() - _T0_SESION) / 60:.1f} min")

liberar()
print("\n⚠️ Antes de cerrar el runtime confirme que las salidas quedaron en Drive:")
print(f"   {RUTA_SALIDA}")

## 🛟 Diagnóstico

| Síntoma | Causa más probable | Qué hacer |
|---|---|---|
| `ModuleNotFoundError: record_linkage` **después** de que pip diga que instaló | Instalación editable: deja un `.pth` que solo se lee al arrancar el intérprete | Ya corregido en la §1 (instalación normal + `invalidate_caches`). Si persiste, reinicie la sesión |
| `Se cargo rues-linker X y este notebook espera 0.22.4` | El kernel tiene una instalación previa cargada | Confirme `RUTA_LIBRERIA`, luego Entorno de ejecución → **Reiniciar sesión** y ejecute la §1 de nuevo |
| `FileNotFoundError` en la §1 | La librería no está en `RUTA_LIBRERIA` | Descomprima el zip y apunte a la carpeta con `pyproject.toml` |
| `[FUENTE] estas columnas declaradas NO existen` | Cambió la maquetación de la entrega | La §4 lista las columnas reales: corrija el mapeo en la §3 |
| `VARIABLES_EXTRA menciona [...]` | Una variable extra no está en todas las fuentes | Mapéela en cada fuente o quítela |
| El enlace es mucho menor de lo esperado | `MUESTRA` activa, o NIT con DV que no cuadra | Ponga `MUESTRA = None`; lea el diagnóstico de la §7 |
| Tarda lo mismo la segunda vez | La caché se invalidó | La §4 dice qué se reutiliza y qué no |
| `transport endpoint is not connected` | Drive usado como disco de trabajo | No pasa aquí: `RUTA_TRABAJO` está en `/content` |
| La sesión se cayó a mitad | Límite de Colab | Vuelva a ejecutar la §6: caché y checkpoints se reutilizan |
| Se quedó sin RAM en "Cargando resultados finales…" | Versión ≤ 0.14.0: L5 retenía varias copias completas del universo | Resuelto desde 0.14.1 (incluido aquí): libera fuentes tras L1, dtypes Arrow y métricas solo sobre pares distintos |
| "Indexando" tarda horas | Versión ≤ 0.14.0: relectura completa del HDF5 por banda + índices parciales | Resuelto: indexación por chunk con índice compuesto único (medido: 1 h 22 min → ~4 min) |
| RAM al límite | Muchas `VARIABLES_EXTRA` o columnas de arrastre pesadas | Quite columnas de la §3 (se re-adjuntan al final igual), baje `MUESTRA` o suba `lsh_threshold` |
| Se fusionaron empresas distintas | Umbrales laxos | Suba `score_threshold` a 0,60 y use `PERFIL = "alta_precision"` |
| Faltan uniones evidentes | Umbrales estrictos | Baje `lsh_threshold` a 0,50 y `score_threshold` a 0,40 |

## 🎚️ Cómo mover los umbrales, con criterio

| Quiero… | Mueva | Costo |
|---|---|---|
| Más recall (encontrar más) | `lsh_threshold` ↓, `score_threshold` ↓ | Más falsos positivos y más tiempo |
| Más precisión (equivocarme menos) | `score_threshold` ↑, `min_name_similarity` ↑ | Se pierden uniones legítimas |
| Ir más rápido | `lsh_permutations` ↓ (128), `lsh_threshold` ↑ | Firma menos fiel, menos recall |
| Nombres muy sucios | `cleaning_mode = "AGRESIVO"` | Puede unir nombres genéricos distintos |
| Nombres cortos o siglas | `cleaning_mode = "CONSERVADOR"` | Menos tolerancia a variantes |
| Evidencia extra (ciudad, teléfono) | Mapéela en la §3 y agréguela a `VARIABLES_EXTRA` | Requiere que exista en todas las fuentes |

**No toque** `veto_nit_base_distinto` ni `cannot_link_identificador` sin una
razón: son las salvaguardas que impiden fusionar empresas con NIT válido
distinto. Medido sobre estos datos, apagarlas contamina 226 grupos.

**Antes de cerrar el runtime**, confirme que las salidas quedaron en Drive
(`RUTA_SALIDA`): el disco local `/content` se pierde con la sesión.